In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ 
\mathbb{R}^n \times \mathbb{R}^n \to \mathbb{R}, \quad y(\mathbf{z}, \mathbf{t}) = 
\frac{1}{N} || \mathbf{z} - \mathbf{t} ||^2 
$$

$$ \text{Derivative} $$

$$ \frac{dy}{dz_i} = \frac{2}{N} (z_i - t_i) $$

$$ \frac{dy}{dz} = \frac{2}{N} (\mathbf{z} - \mathbf{t}) $$

$$ \text{Jacobian} $$

$$
J_y(\mathbf{z}) =
\nabla _y(\mathbf{z}) \top =
\begin{bmatrix}
    \frac{dy}{dz_1}, &
    \frac{dy}{dz_2},  &
    \cdots, &
    \frac{dy}{dz_n}
\end{bmatrix}
$$

$$ d\mathbf{y} = J_y(\mathbf{z}) \, d\mathbf{z} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, d\mathbf{z} \right \rangle =
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle =
\left \langle \frac{dF}{dy}, J_y(\mathbf{z}) \, d\mathbf{z} \right \rangle =
\left \langle J_y(\mathbf{z}) ^\top \, \frac{dF}{dy}, d\mathbf{z} \right \rangle \implies
$$

$$ \frac{dF}{dz} = J_y(\mathbf{z})^\top \, \frac{dF}{dy} = \frac{dy}{dz} \odot \frac{dF}{dy} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def mse(p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
    """Computes the mean squared error between predictions `p` and targets `t`."""
 
    return ((p - t) ** 2).mean()


class MeanSquaredErrorFunction(autograd.Function):
    """Custom autograd function for mean squared error."""

    @staticmethod
    def forward(ctx, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(z, t)
        return mse(z, t)
    
    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
        (z, t) = ctx.saved_tensors
        S = z.shape[0]
        FO = z.shape[1]
        N = S * FO

        dy_dz = 2 / N * (z - t)
        dF_dz = dy_dz * dF_dy
        
        return (dF_dz, None)
    

class MeanSquaredError(nn.Module):
    """Mean squared error loss module."""

    def __init__(self):
        super().__init__()
    
    def forward(self, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return MeanSquaredErrorFunction.apply(z, t)
    

def test_basic():
  z = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32)
  t = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32)

  actual = mse(z, t)
  expected = tr.nn.MSELoss()(z, t)

  assert actual == approx(expected)


def test_gradcheck(samples) -> None:
    def fn(z, t):
        return MeanSquaredErrorFunction.apply(z, t)

    z = tr.randn((samples, 1), dtype=tr.float64, requires_grad=True)
    t = tr.randn((samples, 1), dtype=tr.float64, requires_grad=False)

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, dtype=tr.float32, requires_grad=True)
    t = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    mse_custom = MeanSquaredErrorFunction.apply(z, t)
    mse_builtin = tr.nn.MSELoss()(z, t)

    assert mse_custom == approx(mse_builtin)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)